In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

In [2]:
path = "~/Downloads/AO2Dtree.root"
file = uproot.open(path)

In [3]:
names = file.classnames(filter_classname="TDirectory")
bucket = []

# merge all files and keep:
# - |fPosZ| < 10 cm
# - |eta| < 0.8

max_pos_Z = 10
max_eta = 0.8

collision_ID = "O2collision_001"
track_ID = "O2filtertrack"
extra_ID = "O2filtertrackextr"

for item in names:
    candidate_ID = item
    collision = file[("/".join([candidate_ID,collision_ID]))].arrays(["fPosZ"],library="pd")
    track = file[("/".join([candidate_ID,track_ID]))].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"],library="pd")
    extra = file[("/".join([candidate_ID,extra_ID]))].arrays(["fPt", "fEta", "fCharge", "fDcaXY", "fNsigmaTPCpi", "fNsigmaTPCka",
                                                              "fNsigmaTOFpi", "fNsigmaTOFka"],library="pd")
    complete = ( extra.merge(track, left_index = True, right_index = True) ).merge(collision, left_on ="fIndexCollisions", right_index = True )
    complete = complete[ (np.abs(complete["fPosZ"]) < max_pos_Z ) & ( np.abs(complete["fEta"]) < max_eta ) ]
    complete["file_ID"] = candidate_ID
    bucket.append(complete)
                                                                          
base = pd.concat(bucket, ignore_index=True)

In [11]:
print("The dataframe has", len(base), "rows")
memory = base.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
base

The dataframe has 8784795 rows
The dataframe occupy 1055.61 MB


,fPt,fEta,fCharge,fDcaXY,fNsigmaTPCpi,fNsigmaTPCka,fNsigmaTOFpi,fNsigmaTOFka,fIndexCollisions,fAlpha,fX,fY,fZ,fPosZ,file_ID
0,0.755864,-0.412664,-1,-0.009506,0.747714,-1.404644,2.037047,-20.068079,0,-3.031685,0.030319,0.015100,-4.127120,-4.128410,DF_2300238590827776;1
1,0.663068,0.598317,1,0.004326,1.840203,-0.856808,2.117946,-25.464684,0,-0.322267,-0.017226,-0.030717,-4.129672,-4.128410,DF_2300238590827776;1
2,0.465013,0.462649,-1,0.009618,54.468903,22.347868,-999.000000,-999.000000,0,-3.135348,0.027611,0.037229,-4.130205,-4.128410,DF_2300238590827776;1
3,0.351989,-0.177954,-1,0.013927,-0.477576,-10.363797,-999.000000,-999.000000,0,-2.713416,0.036497,0.027809,-4.132064,-4.128410,DF_2300238590827776;1
4,0.434466,0.289459,-1,0.005372,-0.658317,-7.938647,-0.196841,-26.088795,2,-0.443220,-0.019782,-0.029422,-8.384316,-8.386139,DF_2300238590827776;1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8784790,0.830050,0.670646,1,-0.004397,7.431957,6.485834,-999.000000,-999.000000,3020,1.312377,-0.040394,0.021366,-4.527402,-4.533272,DF_2300238595518368;1
8784791,0.638627,-0.544448,-1,-0.000368,1.678012,-1.493010,-999.000000,-999.000000,3020,-2.320294,0.047770,-0.004037,-4.530087,-4.533272,DF_2300238595518368;1
8784792,0.915340,0.540868,1,-0.005042,-1.053486,-1.164875,-999.000000,-999.000000,3020,1.709269,-0.027295,0.034333,-4.541160,-4.533272,DF_2300238595518368;1
8784793,0.331021,0.484117,-1,-0.022485,0.295924,-10.307919,-999.000000,-999.000000,3020,1.759294,-0.025292,0.018206,-4.530822,-4.533272,DF_2300238595518368;1


In [5]:
collision_f = file[("/".join([candidate_ID,collision_ID]))].arrays(library="pd")

In [6]:
collision_f

,fIndexBCs,fPosX,fPosY,fPosZ,fCovXX,fCovXY,fCovYY,fCovXZ,fCovYZ,fCovZZ,fFlags,fChi2,fNumContrib,fCollisionTime,fCollisionTimeRes
0,2,-0.031304,-0.025318,3.184525,1.309440e-06,-8.544885e-08,1.402572e-06,4.444737e-07,-5.788170e-07,2.386048e-06,0,78.687500,30,-1.718596,7.128906
1,5,-0.039133,-0.027294,-6.039642,1.334585e-06,-9.645009e-08,6.598420e-07,1.576263e-07,1.409353e-08,7.995404e-07,0,76.562500,32,0.119490,4.074219
2,11,-0.036254,-0.025526,-0.006712,4.847534e-07,3.544847e-08,3.678724e-07,1.261651e-08,-2.329762e-08,4.251488e-07,0,100.625000,71,-0.653540,3.265625
3,14,-0.031277,-0.031228,1.633163,2.244487e-06,2.383022e-07,2.508983e-06,-1.181616e-07,5.150214e-07,3.112480e-06,0,27.703125,18,-0.696287,9.984375
4,21,-0.032562,-0.028430,-0.410799,2.961606e-06,5.606562e-07,3.652647e-06,3.168825e-07,6.975606e-07,4.410744e-06,0,20.250000,16,-0.525110,9.968750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3019,15716,-0.026167,-0.029899,-6.388824,2.756715e-06,-1.578592e-07,3.386289e-06,-1.295703e-07,-6.137416e-07,3.531575e-06,0,55.500000,24,0.202779,10.062500
3020,15720,-0.035230,-0.032469,-4.533272,8.777715e-07,-3.396417e-08,8.619390e-07,1.706649e-07,-7.526251e-08,1.092441e-06,0,85.062500,42,-0.202176,7.167969
3021,15728,-0.027489,-0.027814,1.915613,1.111627e-05,9.024516e-07,9.126961e-06,5.424023e-06,4.358590e-06,3.436208e-05,0,19.265625,8,-1.055880,443.000000
3022,15730,-0.031483,-0.025691,-6.384071,2.232194e-05,-3.198162e-06,1.960993e-05,1.958013e-05,-2.078712e-05,1.696348e-04,0,5.699219,3,0.044702,567.500000


In [7]:
track_f = file[("/".join([candidate_ID,track_ID]))].arrays(library="pd")
track_f

,fIndexCollisions,fIsInsideBeamPipe,fTrackType,fX,fAlpha,fY,fZ,fSnp,fTgl,fSigned1Pt
0,-1,1,1,0.043130,-2.186401,-0.019936,0.880172,-1.044564e-07,0.036756,0.727588
1,0,1,1,0.010793,2.522259,-0.079963,-6.203627,-3.168047e-07,0.124881,1.728070
2,0,1,1,-0.035140,1.189931,0.019462,3.182996,-5.150342e-07,0.780761,1.310267
3,0,1,1,0.011959,2.552455,0.032126,3.183084,-7.776002e-07,0.006248,2.196883
4,0,1,1,0.017428,-1.338423,-0.041874,3.184426,5.475188e-07,0.459633,-1.642905
...,...,...,...,...,...,...,...,...,...,...
16636,3020,1,1,-0.040394,1.312377,0.021366,-4.527402,6.492427e-08,0.722061,1.204747
16637,3020,1,1,0.047770,-2.320294,-0.004037,-4.530087,7.594403e-08,-0.571747,-1.565860
16638,3020,1,1,-0.027295,1.709269,0.034333,-4.541160,1.920899e-07,0.567627,1.092490
16639,3020,1,1,-0.025292,1.759294,0.018206,-4.530822,1.260494e-06,0.503251,-3.020955


In [8]:
extra_f = file[("/".join([candidate_ID,extra_ID]))].arrays(library="pd")
extra_f

,fPt,fEta,fCharge,fDcaXY,fDcaZ,fSigmaDcaXY2,fSigmaDcaZ2,fNsigmaTPCpi,fNsigmaTPCka,fNsigmaTPCpr,fNsigmaTOFpi,fNsigmaTOFka,fNsigmaTOFpr
0,1.374405,0.036747,1,-0.009014,0.733283,0.000036,26.378534,0.693235,2.181165,-1.144809,-999.000000,-999.000000,-999.000000
1,0.578680,0.124559,1,-0.118750,-9.388151,0.000029,0.000031,0.195478,-4.632560,-11.159016,1370.390137,163.544067,84.777428
2,0.763203,0.717575,1,-0.000188,-0.001529,0.000021,0.000029,2.348260,1.472154,-4.574846,27.963858,1.615482,-24.834894
3,0.455190,0.006248,1,-0.006318,-0.001441,0.000038,0.000039,1.279183,-6.310167,-12.696817,-0.124970,-41.451733,-84.449364
4,0.608678,0.444818,-1,-0.005580,-0.000098,0.000018,0.000022,-0.674389,-4.222519,-10.849587,-999.000000,-999.000000,-999.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16636,0.830050,0.670646,1,-0.004397,0.005870,0.000022,0.000030,7.431957,6.485834,-1.115745,-999.000000,-999.000000,-999.000000
16637,0.638627,-0.544448,-1,-0.000368,0.003185,0.000021,0.000024,1.678012,-1.493010,-8.809975,-999.000000,-999.000000,-999.000000
16638,0.915340,0.540868,1,-0.005042,-0.007888,0.000014,0.000017,-1.053486,-1.164875,-6.469697,-999.000000,-999.000000,-999.000000
16639,0.331021,0.484117,-1,-0.022485,0.002450,0.000061,0.000073,0.295924,-10.307919,-12.307782,-999.000000,-999.000000,-999.000000


In [9]:
# def to_track_coord( row ):
#     v_in = np.array([row["fPosX"],row["fPosY"]])
#     angle = -row["fAlpha"]
#     rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])  # classical rotation matrix
#     v_out = rot.dot(v_in)

def to_global_coord( row ):
    xy_in = np.array([row["fPosX"],row["fPosY"]])
    angle = +row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])  # classical rotation matrix
    xy_out = rot.dot(xy_in)
    return( xy_out)

- $p_{x}$ and $p_{y}$ in **global** coordinates
- parametrize as straight lines with the converted points and momenta direction
- find the intersection point (approximate X and Y coordinates of the secondary vertex)
- $z_{track, SV}= \frac{p_z}{p_x}x_{SV} + z_0$ with $z_0$ to be determined using the z track coordinate at PCA

In [10]:
# maybe vectorize?
def secondary_vertex ( rows ):
    row1 = rows[0]
    row2 = rows[1]
    x1, y1, x2, y2 = [to_global_coord(row1), to_global_coord(row2)]
    m1 = np.tan( row1["fAlpha"] )
    m2 = np.tan( row2["fAlpha"] )
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
    pz1 = np.senh(row1["fEta"])*row1["fPt"]
    pz2 = np.senh(row2["fEta"])*row2["fPt"]
    z1_track = pz1/(np.cos(row1["fAlpha"])*row1["fPt"]) * x_SV + row1["fZ"]
    z2_track = pz2/(np.cos(row2["fAlpha"])*row2["fPt"]) * x_SV + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return pd.Series([x_SV, y_SV, z_SV])